In [1]:
from dotenv import load_dotenv
load_dotenv()

True

### 1. 문서 로드

In [2]:
# uv add langchain-community pdfplumber
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/Employee_Benefits_Guide_2026_v1.pdf")
documents = loader.load()

### 2. 문서 분할

In [3]:
# uv add langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=100)
texts = text_splitter.split_documents(documents)

### 임베딩 모델(캐싱)

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

store = LocalFileStore("./cache/")

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embeddings,
    store,
    namespace=embeddings.model
)

d:\wsh\langchain-workspace\ch06\.venv\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


### 3. 임베딩 & FAISS 벡터스토어 생성 및 저장

In [5]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(texts, cached_embedder) # in-memory

In [7]:
vector_store.save_local("./faiss_index") # 로컬 디스크에 저장하기

In [8]:
# 벡터스토어 재로딩
vectorstore = FAISS.load_local(
    "./faiss_index", # 저장된 FAISS 인덱스 폴더의 경로
    cached_embedder,
    allow_dangerous_deserialization=True, #  FAISS 인덱스 내 데이터 역직렬화(deserialization) 허용(신뢰할 수 있는 파일 일 경우)
)

In [13]:
query = "이번 주에 내가 결혼을 하는데 얼마를 받을 수 있을까?"

In [15]:
response = vectorstore.similarity_search(query, k=1)

In [16]:
response

[Document(id='f001c508-9324-42cb-b297-ce2545bd37b9', metadata={'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'file_path': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'page': 11, 'total_pages': 22, 'Author': '', 'CreationDate': "D:20260116002528+09'00'", 'ModDate': "D:20260116002528+09'00'", 'Producer': 'Microsoft: Print To PDF', 'Title': 'Microsoft Word - Employee_Benefits_Guide_2026_v1.docx'}, page_content='\uf0b7 소멸: 당해 연도 12 월 31 일까지 미사용 시 자동 소멸 (이월 불가).\n5.4 경조사 지원 기준 (Family Support Details)\n기쁨과 슬픔을 함께 나누는 테크노빌드의 경조 지원입니다.\n구분 대상 휴가 (일수) 경조금 (만원) 화환/조화\n결혼 본인 5 일 100 + 화환 지원\n자녀 1 일 50 + 화환 지원\n30 -\n형제/자매 1 일\n30 -\n회갑/칠순 본인/배우자 부모 1 일\n출산 본인 출산휴가 (90 일) 출산 축하금 50 과일 바구니\n배우자 10 일 (유급) 출산 축하금 50 과일 바구니\n-\n사망 본인/배우자 500 + 장례용품 3 단 조화 + 근조기\n부모/배우자 부모 5 일 100 + 장례용품 3 단 조화 + 근조기\n30\n조부모/외조부모 3 일 조화\n30\n형제/자매 3 일 조화')]

In [11]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """다음 컨텍스트만 사용해 질문에 답하세요.

- 컨텍스트: {context}
- 질문: {question}"""
)

In [12]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

model = init_chat_model("google_genai:gemini-2.5-flash-lite")

chain = prompt | model | StrOutputParser()

In [19]:
result = chain.invoke({
    "context": response,
    "question": query
})

In [20]:
result

'이번 주에 결혼하신다면 본인 결혼 축하금으로 100만원과 화환을 지원받으실 수 있습니다.'

In [21]:
retriver = vectorstore.as_retriever(k=1)

In [ ]:
retriver.invoke(query)

In [27]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

model = init_chat_model("google_genai:gemini-2.5-flash-lite")

chain = (
    {"context": retriver, "question": RunnablePassthrough()}
    | prompt 
    | model 
    | StrOutputParser()
)

In [28]:
result = chain.invoke(query)

In [29]:
print(result)

컨텍스트에 따르면, 본인 결혼 시 100만원과 화환이 지원됩니다.
